# LLM Exposure Index – Datenerhebung und Aufbereitung
**Masterarbeit | Kapitel 5.1 – 5.2.4**  
Autor: Ayumi Nojima  
Datum: April 2026  

**Datenquellen:**
- O\*NET® 30.2 Database – U.S. Department of Labor (Abilities, Skills, Work Activities, Knowledge)
- CH-ISCO-19 / BFS Strukturerhebung – Bundesamt für Statistik (CH_ISCO19.xlsx)
- ILO SOC→ISCO-08 Crosswalk (isco_soc_crosswalk.xls)

**Dieses Notebook deckt folgende Schritte ab:**
- **5.1.1** O\*NET-Fähigkeitsprofile: Laden, Filtern (Scale ID = IM), Bereinigen, Normalisieren
- **5.1.2** CH-ISCO-19: Extraktion der 4-stelligen Codes für Hauptgruppen 1–3
- **5.1.3** BFS-Beschäftigungsdaten: ΔBFS_j-Berechnung (2022→2024), Ausreissererkennung
- **5.2.1** Mapping O\*NET → CH-ISCO-19 (dreistufig: hierarchisch, Fuzzy, manuell)
- **5.2.2** Kalibrierung der LLM-Gewichte μ_i
- **5.2.3** Berechnung des Exposure-Index E_j, E^sub_j, E^aug_j
- **5.2.4** Vorbereitung der Analysedatensätze (H1, H2, H3, Validierung)

**Repository-Struktur:**
```
data/
├── raw/
│   ├── onet/          # O*NET 30.2 xlsx-Dateien
│   ├── CH_ISCO19.xlsx
│   └── isco_soc_crosswalk.xls
├── processed/
└── output/
```

In [4]:
from pathlib import Path

# Repo-Root = zwei Ebenen über diesem Notebook
REPO_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent

RAW_ONET  = REPO_ROOT / "data" / "raw" / "onet"
RAW_OTHER = REPO_ROOT / "data" / "raw"
PROCESSED = REPO_ROOT / "data" / "processed"
OUTPUT    = REPO_ROOT / "data" / "output"

for p in [PROCESSED, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Root:", REPO_ROOT)
print("Processed:", PROCESSED)

Root: /workspaces/Master_Thesis
Processed: /workspaces/Master_Thesis/data/processed


## 0. Imports und Konfiguration

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from rapidfuzz import fuzz, process
import warnings
warnings.filterwarnings('ignore')

# ── Pfade ──────────────────────────────────────────────────────────────────
RAW_ONET  = Path("data/raw/onet")
RAW_OTHER = Path("data/raw")
PROCESSED = Path("data/processed")
OUTPUT    = Path("data/output")
for p in [PROCESSED, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Parameter (vgl. Kap. 4.5 / 4.9) ───────────────────────────────────────
MISSING_VALUE_THRESHOLD = 0.10   # Max. 10% fehlende Werte je Beruf (Newman, 2014)
MIN_EMPLOYMENT          = 50     # Mindestbeschäftigte 2022 in absoluten Zahlen
SKILL_OVERLAP_THRESHOLD = 0.70   # Pearson-Korrelation s_ij-Mindestwert
FUZZY_SCORE_MIN         = 80     # RapidFuzz token_sort_ratio Mindestscore (0–100)
MAIN_GROUPS             = [1, 2, 3]  # Wissensintensive Berufe CH-ISCO-19
BFS_BASE_YEAR           = 2022
BFS_TARGET_YEAR         = 2024
RANDOM_SEED             = 42
N_MC_ITERATIONS         = 1000   # Monte-Carlo-Iterationen

print("Konfiguration geladen ✓")

Konfiguration geladen ✓


---
## 5.1.1 O\*NET-Fähigkeitsprofile

Die O\*NET 30.2-Daten werden aus vier Teildatensätzen geladen: **Abilities, Skills, Work Activities, Knowledge**.  
Für die Exposure-Berechnung wird ausschliesslich die **Importance-Dimension (Scale ID = "IM")** verwendet,  
da sie den relativen Stellenwert einer Fähigkeit im Berufsalltag quantifiziert  
(Eloundou et al., 2023; Kuprecht, 2025).

**Wichtig:** Der Dimensionsname wird mit dem Quelldatensatz als Präfix versehen (`Abilities – Oral Comprehension`),  
um Namenskollisionen zu vermeiden (z.B. "Mathematics" erscheint in Skills *und* Knowledge).

In [6]:
def load_onet_file(filename: str, source_label: str) -> pd.DataFrame:
    """
    Lädt eine O*NET-Datei, filtert auf Importance-Dimension (Scale ID = IM)
    und prefixed den Element-Namen mit dem Quelldatensatz.
    
    Parameters
    ----------
    filename : str
        Dateiname relativ zu RAW_ONET.
    source_label : str
        Präfix für Element-Namen (z.B. 'Abilities', 'Skills').
    
    Returns
    -------
    pd.DataFrame mit Spalten: soc_code, soc_title, element_id,
                               element_name (prefixed), importance, source
    """
    df = pd.read_excel(RAW_ONET / filename)
    df.columns = df.columns.str.strip()
    df = df[df["Scale ID"] == "IM"].copy()
    df = df[["O*NET-SOC Code", "Title", "Element ID",
             "Element Name", "Data Value"]].copy()
    df.rename(columns={
        "O*NET-SOC Code": "soc_code",
        "Title":          "soc_title",
        "Element ID":     "element_id",
        "Element Name":   "element_name",
        "Data Value":     "importance"
    }, inplace=True)
    # Präfix verhindert Namenskollisionen zwischen Datensätzen
    df["element_name"] = source_label + " – " + df["element_name"]
    df["source"] = source_label
    return df


abilities  = load_onet_file("/workspaces/Master_Thesis/data/raw/onet/Abilities.xlsx",      "Abilities")
skills     = load_onet_file("/workspaces/Master_Thesis/data/raw/onet/Skills.xlsx",         "Skills")
work_act   = load_onet_file("/workspaces/Master_Thesis/data/raw/onet/Work Activities.xlsx", "Work Activities")
knowledge  = load_onet_file("/workspaces/Master_Thesis/data/raw/onet/Knowledge.xlsx",      "Knowledge")

onet_raw = pd.concat([abilities, skills, work_act, knowledge], ignore_index=True)

print(f"O*NET Rohdaten: {len(onet_raw):,} Zeilen")
print(f"  Berufe:      {onet_raw['soc_code'].nunique()}")
print(f"  Dimensionen: {onet_raw['element_name'].nunique()}")
print(f"  Quellen:     {onet_raw['source'].unique().tolist()}")

O*NET Rohdaten: 143,934 Zeilen
  Berufe:      894
  Dimensionen: 161
  Quellen:     ['Abilities', 'Skills', 'Work Activities', 'Knowledge']


### Bereinigung der O\*NET-Daten

Drei Bereinigungsschritte gemäss Kap. 5.1.1:
1. Ausschluss von Berufen mit >10% fehlenden Werten (Newman, 2014)
2. Median-Imputation für verbleibende Fehlwerte je Dimension
3. Min-Max-Normalisierung aller Importance-Werte auf [0,1] je Dimension

In [7]:
total_dims = onet_raw["element_name"].nunique()

# Schritt 1: Fehlwertquote je Beruf prüfen
missing_by_occ = (
    onet_raw.groupby("soc_code")["importance"]
    .apply(lambda x: x.isna().sum() / total_dims)
    .reset_index(name="missing_rate")
)
excluded = missing_by_occ[missing_by_occ["missing_rate"] > MISSING_VALUE_THRESHOLD]
valid_soc = missing_by_occ[missing_by_occ["missing_rate"] <= MISSING_VALUE_THRESHOLD]["soc_code"]
print(f"Ausgeschlossen (>{MISSING_VALUE_THRESHOLD*100:.0f}% fehlende Werte): {len(excluded)} Berufe")

onet_clean = onet_raw[onet_raw["soc_code"].isin(valid_soc)].copy()

# Schritt 2: Median-Imputation je Dimension
onet_clean["importance"] = onet_clean["importance"].fillna(
    onet_clean.groupby("element_name")["importance"].transform("median")
)

# Schritt 3: Min-Max-Normalisierung → w_ij in [0,1]
onet_clean["w_ij"] = onet_clean.groupby("element_name")["importance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

print(f"O*NET nach Bereinigung: {onet_clean['soc_code'].nunique()} Berufe | "
      f"{onet_clean['element_name'].nunique()} Dimensionen")

# Pivot → Berufs-Fähigkeits-Matrix (Zeilen=Berufe, Spalten=Dimensionen)
onet_pivot = onet_clean.pivot_table(
    index="soc_code", columns="element_name", values="w_ij", aggfunc="mean"
).reset_index()

print(f"Pivot-Matrix: {onet_pivot.shape[0]} Berufe × {onet_pivot.shape[1]-1} Fähigkeitsdimensionen")
onet_pivot.to_csv(PROCESSED / "onet_pivot.csv", index=False)
print("Gespeichert → data/processed/onet_pivot.csv")

Ausgeschlossen (>10% fehlende Werte): 0 Berufe
O*NET nach Bereinigung: 894 Berufe | 161 Dimensionen
Pivot-Matrix: 894 Berufe × 161 Fähigkeitsdimensionen
Gespeichert → data/processed/onet_pivot.csv


---
## 5.1.2 CH-ISCO-19-Klassifikation

Die CH-ISCO-19-Klassifikation wird aus `CH_ISCO19.xlsx` extrahiert.  
Die Datei enthält mehrere Sheets; für die Klassifikation wird das Sheet `2022-2024` verwendet,  
das die poolten Beschäftigungsdaten (kumuliert) enthält.  

**Struktur des Sheets** (Spaltenindex):
- Spalte 0: Berufshauptgruppe (1-stellig)
- Spalte 1: Berufsgruppe (2-stellig)
- Spalte 2: Berufsuntergruppe (3-stellig)
- Spalte 3: Berufsgattung (4-stellig) ← **Analyseebene**
- Spalte 4: Berufsart (5-stellig)
- Spalte 5: Berufsbezeichnung
- Spalte 6: Total Beschäftigte (in 1000)

Für die Analyse werden **nur Hauptgruppen 1, 2, 3** berücksichtigt (Kap. 4.1).  
Berufsbezeichnungen werden für das Fuzzy-Matching harmonisiert.

In [8]:
def extract_chisco(sheet_name: str) -> pd.DataFrame:
    """
    Extrahiert 4-stellige CH-ISCO-19-Codes für Hauptgruppen 1–3.
    
    Spaltenstruktur des BFS-Sheets (0-indexiert):
      0=Hauptgruppe, 1=Berufsgruppe, 2=Berufsuntergruppe,
      3=Berufsgattung (4-dig), 4=Berufsart, 5=Bezeichnung, 6=Total
    
    Returns
    -------
    pd.DataFrame mit: ch_isco_4digit, label, main_group, label_clean
    """
    df = pd.read_excel(REPO_ROOT / "data" / "raw" / "CH_ISCO19.xlsx",
                       sheet_name="2022-2024", header=None)

    # Hauptgruppe aus Spalte 0 forward-fillen
    df["hg"] = pd.to_numeric(df[0], errors="coerce").ffill()

    # 4-stellige Codes in Spalte 3
    mask = (
        df[3].notna() &
        df[3].astype(str).str.match(r"^\d{4}$") &
        df["hg"].isin(MAIN_GROUPS)
    )
    result = df[mask][[3, 5, "hg"]].copy()
    result.columns = ["ch_isco_4digit", "label", "main_group"]
    result["ch_isco_4digit"] = result["ch_isco_4digit"].astype(str)
    result["main_group"] = result["main_group"].astype(int)

    # Bezeichnungen harmonisieren für Fuzzy-Matching (Stufe 2)
    result["label_clean"] = (
        result["label"]
        .str.strip().str.lower()
        .str.replace(r"ä", "ae", regex=False)
        .str.replace(r"ö", "oe", regex=False)
        .str.replace(r"ü", "ue", regex=False)
        .str.replace(r"[^a-z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return result.reset_index(drop=True)


ch_isco = extract_chisco("2022-2024")

print(f"CH-ISCO-19 Hauptgruppen 1–3: {len(ch_isco)} Berufsklassen")
print(f"  HG 1 (Führungskräfte):        {(ch_isco['main_group']==1).sum()}")
print(f"  HG 2 (Akademische Berufe):    {(ch_isco['main_group']==2).sum()}")
print(f"  HG 3 (Techniker etc.):        {(ch_isco['main_group']==3).sum()}")

ch_isco.to_csv(PROCESSED / "ch_isco_clean.csv", index=False)
print("Gespeichert → data/processed/ch_isco_clean.csv ✓")
ch_isco.head(5)

CH-ISCO-19 Hauptgruppen 1–3: 271 Berufsklassen
  HG 1 (Führungskräfte):        44
  HG 2 (Akademische Berufe):    119
  HG 3 (Techniker etc.):        108
Gespeichert → data/processed/ch_isco_clean.csv ✓


,ch_isco_4digit,label,main_group,label_clean
0,1000,"Führungskräfte, onA",1,fuehrungskraefte ona
1,1100,"Geschäftsführer, Vorstände, leitende Verwaltun...",1,geschaeftsfuehrer vorstaende leitende verwaltu...
2,1110,Angehörige gesetzgebender Körperschaften und l...,1,angehoerige gesetzgebender koerperschaften und...
3,1111,Angehörige gesetzgebender Körperschaften,1,angehoerige gesetzgebender koerperschaften
4,1112,Leitende Verwaltungsbedienstete,1,leitende verwaltungsbedienstete


---
## 5.1.3 BFS-Strukturerhebung (ΔBFS_j)

Die Beschäftigungsdaten werden aus den Einzeljahressheets `2022` und `2024` extrahiert.

Die abhängige Variable für H3 wird gemäss Kap. 4.4 berechnet:

$$\Delta BFS_j = \frac{\text{Beschäftigte}_{2024} - \text{Beschäftigte}_{2022}}{\text{Beschäftigte}_{2022}} \times 100$$

**Bereinigungsschritte:**
- Ausschluss von Berufsgruppen mit <50 Beschäftigten im Basisjahr 2022
- Supprimierte BFS-Werte ('X') werden als NaN behandelt
- Ausreissererkennung via IQR-Methode (Hastie et al., 2009)

*Hinweis: BFS-Beschäftigungszahlen sind in 1000 angegeben. Die Mindestfallzahl von 50 entspricht 0.05 in der Originalskalierung.*

In [9]:
def extract_bfs_year(sheet_name: str, year_label: str) -> pd.DataFrame:
    """
    Extrahiert Beschäftigtenzahlen (Total, in 1000) für ein einzelnes Jahr
    aus CH_ISCO19.xlsx für Hauptgruppen 1–3, 4-stellige Codes.

    sheet_name : Name des Sheets (z.B. '2022', '2024')
    year_label : Label für die Zielspalte (z.B. 'employed_2022')
    Supprimierte Werte ('X') werden als NaN kodiert.
    """
    df = pd.read_excel(
        REPO_ROOT / "data" / "raw" / "CH_ISCO19.xlsx",
        sheet_name=sheet_name,
        header=None
    )

    # Hauptgruppe via ffill aus Spalte 0 ableiten
    df["hg"] = pd.to_numeric(df[0], errors="coerce").ffill()

    # Nur 4-stellige Berufsgattungs-Codes der Hauptgruppen 1–3
    col3_str = df[3].astype(str)
    mask = (
        col3_str.str.match(r"^\d{4}$") &
        df["hg"].isin([1.0, 2.0, 3.0])
    )

    result = df[mask][[3, 6]].copy()
    result.columns = ["ch_isco_4digit", year_label]
    result["ch_isco_4digit"] = result["ch_isco_4digit"].astype(str)

    # 'X' (BFS-Geheimhaltung) → NaN
    result[year_label] = pd.to_numeric(result[year_label], errors="coerce")

    return result.reset_index(drop=True)


# Korrekt: einzelne Jahres-Sheets verwenden, nicht die gepoolten
bfs22 = extract_bfs_year("2022", "employed_2022")
bfs24 = extract_bfs_year("2024", "employed_2024")

# Merge beider Jahre
bfs = bfs22.merge(bfs24, on="ch_isco_4digit", how="inner")

# Mindestfallzahl: 50 Beschäftigte = 0.05 (in 1000)
MIN_EMP_SCALED = MIN_EMPLOYMENT / 1000
n_before = len(bfs)
bfs = bfs.dropna(subset=["employed_2022", "employed_2024"])
bfs = bfs[bfs["employed_2022"] >= MIN_EMP_SCALED].copy()
print(f"Ausgeschlossen (<{MIN_EMPLOYMENT} Beschäftigte oder NaN 2022): "
      f"{n_before - len(bfs)} Berufsgruppen")

# ΔBFS_j berechnen
bfs["delta_bfs"] = (
    (bfs["employed_2024"] - bfs["employed_2022"]) / bfs["employed_2022"] * 100
)

# Ausreissererkennung via IQR
Q1, Q3 = bfs["delta_bfs"].quantile([0.25, 0.75])
IQR = Q3 - Q1
bfs["is_outlier"] = ~bfs["delta_bfs"].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

print(f"BFS-Datensatz: {len(bfs)} Berufsgruppen | "
      f"Ausreisser (IQR): {bfs['is_outlier'].sum()}")
print(f"ΔBFS-Verteilung: Mean={bfs['delta_bfs'].mean():.1f}% | "
      f"Std={bfs['delta_bfs'].std():.1f}% | "
      f"Min={bfs['delta_bfs'].min():.1f}% | Max={bfs['delta_bfs'].max():.1f}%")

bfs.to_csv(PROCESSED / "bfs_clean.csv", index=False)
print("Gespeichert → data/processed/bfs_clean.csv ✓")

Ausgeschlossen (<50 Beschäftigte oder NaN 2022): 56 Berufsgruppen
BFS-Datensatz: 215 Berufsgruppen | Ausreisser (IQR): 15
ΔBFS-Verteilung: Mean=6.6% | Std=23.5% | Min=-52.6% | Max=156.7%
Gespeichert → data/processed/bfs_clean.csv ✓


### Ausgangslage vor dem Mapping

In [16]:
print("=" * 55)
print("AUSGANGSLAGE VOR MAPPING")
print("=" * 55)
print(f"  O*NET-Berufe (bereinigt):        {onet_pivot.shape[0]:>5}")
print(f"  CH-ISCO-19 Klassen (HG 1–3):     {len(ch_isco):>5}")
print(f"  BFS-Berufsgruppen (bereinigt):   {len(bfs):>5}")
print(f"  Ziel-Sample (Kap. 4):            ~100")
print("=" * 55)

AUSGANGSLAGE VOR MAPPING
  O*NET-Berufe (bereinigt):          894
  CH-ISCO-19 Klassen (HG 1–3):       271
  BFS-Berufsgruppen (bereinigt):     215
  Ziel-Sample (Kap. 4):            ~100


---
## 5.2.1 Mapping O\*NET → CH-ISCO-19 (dreistufig)

Das Mapping verbindet amerikanische O\*NET-SOC-Codes mit Schweizer CH-ISCO-19-Klassen  
über ein **dreistufiges hierarchisches Verfahren** (vgl. Kap. 4.7):

| Stufe | Methode | Kriterium |
|-------|---------|------|
| 1 | Hierarchisch via ILO-Korrespondenztabelle | SOC → ISCO-08 → CH-ISCO-19 |
| 2 | Fuzzy-Matching (RapidFuzz token_sort_ratio) | Score ≥ 80 |
| 3 | Manuelle Korrektur (zwei Kodierer, Krippendorff-α > 0.80) | Abweichung > 10% |

In [10]:
# ── Zelle 1: ESCO→O*NET Crosswalk laden ───────────────────────────────────
# Quelle: O*NET Center — ESCO to O*NET-SOC 2019 Crosswalk
# Datei: data/raw/ESCO_to_ONET-SOC.xlsx (manuell heruntergeladen von onetcenter.org)

esco_raw = pd.read_excel("/workspaces/Master_Thesis/data/raw/ESCO_to_ONET-SOC.xlsx", header=2)
esco_raw.columns = esco_raw.iloc[0].tolist()
esco_raw = esco_raw.iloc[1:].reset_index(drop=True)
esco_raw.columns = [str(c).strip() for c in esco_raw.columns]

# ISCO-4-digit aus ESCO/ISCO Code extrahieren (Format: "1234.56" → "1234")
esco_raw["isco_4digit"] = esco_raw["ESCO/ISCO Code"].astype(str).str.extract(r"^(\d{4})")
esco_raw["isco_hg"]     = pd.to_numeric(esco_raw["isco_4digit"].str[:1], errors="coerce")

print(f"ESCO Crosswalk geladen: {len(esco_raw):,} Einträge")
print(f"ISCO Hauptgruppen vorhanden: {sorted(esco_raw['isco_hg'].dropna().unique().astype(int))}")
print(f"Einträge HG 1–3: {esco_raw['isco_hg'].isin([1,2,3]).sum():,}")

ESCO Crosswalk geladen: 8,629 Einträge
ISCO Hauptgruppen vorhanden: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Einträge HG 1–3: 5,359


In [ ]:
# ─Stufe 1 – Hierarchisches Matching via ESCO-Crosswalk ──────────
# Strategie: pro O*NET-SOC Code den häufigsten ISCO-4-digit Code wählen
# Bei Gleichstand: niedrigerer ISCO-Code bevorzugt (deterministisch)
# Quelle: ESCO to O*NET-SOC 2019 (O*NET Center / EU-Kommission)

hg123 = esco_raw[esco_raw["isco_hg"].isin([1.0, 2.0, 3.0])].copy()

# Eindeutiges 1:1 Mapping: häufigster ISCO-Code je O*NET-Code
best_map = (
    hg123.groupby(["O*NET-SOC 2019 Code", "isco_4digit"])
    .size()
    .reset_index(name="count")
    .sort_values(["count", "isco_4digit"], ascending=[False, True])
    .drop_duplicates("O*NET-SOC 2019 Code")
    [["O*NET-SOC 2019 Code", "isco_4digit"]]
    .rename(columns={"O*NET-SOC 2019 Code": "soc_code"})
)

# O*NET-Pivot mit Mapping mergen
# O*NET-SOC Codes: 11-1011.00 Format — direkt kompatibel
stage1 = onet_pivot[["soc_code"]].merge(best_map, on="soc_code", how="left")

mapped_s1   = stage1.dropna(subset=["isco_4digit"]).copy()
unmapped_s1 = stage1[stage1["isco_4digit"].isna()]["soc_code"].values

print(f"Stufe 1 – Hierarchisches Matching via ESCO-Crosswalk:")
print(f"  Zugeordnet:    {len(mapped_s1)} Berufe")
print(f"  Offen für S2:  {len(unmapped_s1)} Berufe")
print(f"  Abdeckung:     {len(mapped_s1)/len(onet_pivot)*100:.1f}%")

mapped_s1.to_csv(PROCESSED / "stage1_mapping.csv", index=False)
print("\nGespeichert → data/processed/stage1_mapping.csv ✓")

Stufe 1 – Hierarchisches Matching via ESCO-Crosswalk:
  Zugeordnet:    714 Berufe
  Offen für S2:  180 Berufe
  Abdeckung:     79.9%

Gespeichert → data/processed/stage1_mapping.csv ✓


In [16]:
# ── STUFE 2: Fuzzy-Matching (RapidFuzz token_sort_ratio) ──────────────────
# Für nicht zugeordnete Berufe: Bezeichnungsähnlichkeit zwischen
# O*NET-Titeln (englisch) und CH-ISCO-19-Labels (deutsch, harmonisiert).
# Hinweis: Cross-language matching hat begrenzte Trennschärfe;
# Stufe 2 dient primär der Abdeckungserhöhung, Stufe 3 sichert Qualität.

occ_data = pd.read_excel("/workspaces/Master_Thesis/data/raw/onet/Occupation Data.xlsx")
occ_data.columns = occ_data.columns.str.strip()
occ_data.rename(columns={"O*NET-SOC Code": "soc_code",
                          "Title":          "onet_title"}, inplace=True)
occ_data = occ_data[["soc_code", "onet_title"]].copy()

# Nur Berufe die in Stufe 1 NICHT zugeordnet wurden
unmapped_df = occ_data[occ_data["soc_code"].isin(unmapped_s1)].copy()
unmapped_df["title_clean"] = (
    unmapped_df["onet_title"]
    .str.strip().str.lower()
    .str.replace(r"[^a-z0-9 ]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

# Lookup-Dictionary: {label_clean: ch_isco_4digit}
ch_isco_lookup  = dict(zip(ch_isco["label_clean"], ch_isco["ch_isco_4digit"]))
ch_isco_choices = list(ch_isco_lookup.keys())

def fuzzy_match_onet(title: str) -> tuple:
    """Gibt (ch_isco_4digit, score) zurück oder (None, None) bei kein Match."""
    result = process.extractOne(
        title, ch_isco_choices,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=FUZZY_SCORE_MIN
    )
    if result:
        return ch_isco_lookup[result[0]], result[1]
    return None, None

mapped_s2_records = []
for _, row in unmapped_df.iterrows():
    code, score = fuzzy_match_onet(row["title_clean"])
    if code:
        mapped_s2_records.append({
            "soc_code":       row["soc_code"],
            "isco_4digit":    code,          # ← einheitlich mit Stufe 1
            "fuzzy_score":    score,
            "mapping_stage":  2
        })

mapped_s2   = pd.DataFrame(mapped_s2_records)
unmapped_s2 = [s for s in unmapped_s1
               if len(mapped_s2) == 0 or s not in mapped_s2["soc_code"].values]

print(f"Stufe 2 – Fuzzy-Matching (Score ≥ {FUZZY_SCORE_MIN}):")
print(f"  Zugeordnet:    {len(mapped_s2):>4} Berufe")
print(f"  Offen für S3:  {len(unmapped_s2):>4} Berufe (manuelle Prüfung)")

Stufe 2 – Fuzzy-Matching (Score ≥ 80):
  Zugeordnet:       0 Berufe
  Offen für S3:   180 Berufe (manuelle Prüfung)


In [20]:
# ── STUFE 3: Manuelle Korrektur ───────────────────────────────────────────
# Verbleibende Berufe ohne automatische Zuordnung → CSV-Export für Kodierer.
# Zwei unabhängige Kodierer; Krippendorff-α Ziel: > 0.80

manual_review = occ_data[occ_data["soc_code"].isin(unmapped_s2)].copy()
manual_review["isco_4digit_coder1"] = ""
manual_review["isco_4digit_coder2"] = ""
manual_review["isco_4digit_final"]  = ""

manual_path = PROCESSED / "stage3_manual_review.csv"
manual_review.to_csv(manual_path, index=False)
print(f"Stufe 3 – Manuelle Prüfung:")
print(f"  {len(manual_review)} Berufe exportiert → {manual_path}")
print(f"  → Datei befüllen und als 'stage3_completed.csv' speichern")

Stufe 3 – Manuelle Prüfung:
  180 Berufe exportiert → data/processed/stage3_manual_review.csv
  → Datei befüllen und als 'stage3_completed.csv' speichern


In [19]:
# ── Mapping zusammenführen ────────────────────────────────────────────────
# Stufe 3 einlesen (nach manueller Kodierung), sonst leer weiterfahren

try:
    mapped_s3 = pd.read_csv(PROCESSED / "stage3_completed.csv")
    mapped_s3 = (mapped_s3[["soc_code", "isco_4digit_final"]]
                 .rename(columns={"isco_4digit_final": "isco_4digit"}))
    mapped_s3["mapping_stage"] = 3
    print(f"Stufe 3 geladen: {len(mapped_s3)} Berufe")
except FileNotFoundError:
    print("HINWEIS: stage3_completed.csv nicht vorhanden – Stufe 3 übersprungen")
    mapped_s3 = pd.DataFrame(columns=["soc_code", "isco_4digit", "mapping_stage"])

# Alle drei Stufen kombinieren — Stufe 1 hat Priorität
mapped_s1_df = mapped_s1[["soc_code", "isco_4digit"]].copy()
mapped_s1_df["mapping_stage"] = 1

mapping_all = (
    pd.concat([mapped_s1_df,
               mapped_s2[["soc_code", "isco_4digit", "mapping_stage"]],
               mapped_s3],
              ignore_index=True)
    .drop_duplicates("soc_code", keep="first")
)

# Nur Hauptgruppen 1–3 behalten
mapping_all = mapping_all[
    mapping_all["isco_4digit"].astype(str).str[:1].isin(["1", "2", "3"])
].copy()

stage_counts = mapping_all["mapping_stage"].value_counts().sort_index()
print("\nMapping-Gesamtergebnis:")
for stage, count in stage_counts.items():
    print(f"  Stufe {int(stage)}: {count:>4} Berufe")
print(f"  Total:  {len(mapping_all):>4} Berufe gemappt")
print(f"  Abdeckung O*NET HG 1–3: {len(mapping_all)/len(onet_pivot)*100:.1f}%")

mapping_all.to_csv(PROCESSED / "onet_chisco_mapping.csv", index=False)
print("\nGespeichert → data/processed/onet_chisco_mapping.csv ✓")

HINWEIS: stage3_completed.csv nicht vorhanden – Stufe 3 übersprungen


KeyError: "None of [Index(['soc_code', 'isco_4digit', 'mapping_stage'], dtype='str')] are in the [columns]"

### Hold-out-Validierung (20%)

20% der gemappten Berufe werden als Hold-out-Set zurückgehalten,  
um die Generalisierbarkeit des Mapping-Verfahrens zu prüfen (Hastie et al., 2009).

In [ ]:
train_mapping, holdout_mapping = train_test_split(
    mapping_all, test_size=0.20, random_state=RANDOM_SEED
)
holdout_mapping.to_csv(PROCESSED / "holdout_mapping.csv", index=False)
train_mapping.to_csv(PROCESSED / "train_mapping.csv", index=False)

# Übereinstimmungsrate auf Hold-out nach Stufe
s1_rate   = (holdout_mapping["mapping_stage"] == 1).mean() * 100
s1s2_rate = holdout_mapping["mapping_stage"].isin([1, 2]).mean() * 100

print(f"Hold-out-Set: {len(holdout_mapping)} Berufe (20%)")
print(f"  Abdeckung Stufe 1 allein:    {s1_rate:.1f}%")
print(f"  Abdeckung Stufen 1+2:        {s1s2_rate:.1f}%")
print("Gespeichert → data/processed/train_mapping.csv ✓")
print("Gespeichert → data/processed/holdout_mapping.csv ✓")

---
## 5.2.2 Kalibrierung der LLM-Gewichte (μ_i)

Die LLM-Gewichte μ_i basieren auf den Exposure-Bewertungen aus **Eloundou et al. (2023)**  
und wurden für aktuelle Benchmark-Daten selektiv adjustiert  
(OpenAI et al., 2024; DeepSeek-AI et al., 2025) – insbesondere in den Bereichen  
komplexes Schlussfolgern, Textgenerierung und Codegenerierung.

**Schwellenwerte (Cohen, 1988; Eloundou et al., 2023):**
- μ_i > 0.7 → Substitutionsanteil (E^sub_j)
- 0.3 ≤ μ_i ≤ 0.7 → Augmentationsanteil (E^aug_j)
- μ_i < 0.3 → nicht in Index aufgenommen (vernachlässigbarer LLM-Bezug)

Alle Gewichte werden auf **[0.1, 0.9]** rescaled, um Extremnullgewichtungen zu vermeiden.

**Hinweis:** Die Dimensionsnamen verwenden den Präfix aus Abschnitt 5.1.1 (`Abilities – ...` etc.),
um eindeutige Zuordnung zu gewährleisten.

In [ ]:
# LLM-Gewichte je Fähigkeitsdimension (Eloundou et al. 2023, adjustiert)
# Skala: 0=kein LLM-Bezug, 1=vollständig durch LLM abdeckbar
# Präfix muss mit load_onet_file()-Output übereinstimmen
mu_weights_raw = {
    # ── Abilities ──────────────────────────────────────────────────────────
    "Abilities – Oral Comprehension":       0.82,
    "Abilities – Written Comprehension":    0.88,
    "Abilities – Oral Expression":          0.70,
    "Abilities – Written Expression":       0.88,
    "Abilities – Fluency of Ideas":         0.72,
    "Abilities – Originality":              0.65,
    "Abilities – Problem Sensitivity":      0.78,
    "Abilities – Deductive Reasoning":      0.85,
    "Abilities – Inductive Reasoning":      0.82,
    "Abilities – Information Ordering":     0.80,
    "Abilities – Category Flexibility":     0.72,
    "Abilities – Mathematical Reasoning":   0.80,
    "Abilities – Number Facility":          0.82,
    "Abilities – Memorization":             0.75,
    "Abilities – Speed of Closure":         0.55,
    "Abilities – Perceptual Speed":         0.45,
    "Abilities – Spatial Orientation":      0.20,
    "Abilities – Visualization":            0.35,
    "Abilities – Selective Attention":      0.45,
    "Abilities – Time Sharing":             0.30,
    "Abilities – Arm-Hand Steadiness":      0.05,
    "Abilities – Manual Dexterity":         0.05,
    "Abilities – Finger Dexterity":         0.05,
    "Abilities – Near Vision":              0.10,
    "Abilities – Far Vision":               0.05,
    "Abilities – Speech Recognition":       0.60,
    "Abilities – Speech Clarity":           0.50,
    # ── Skills ─────────────────────────────────────────────────────────────
    "Skills – Reading Comprehension":       0.85,
    "Skills – Active Listening":            0.55,
    "Skills – Writing":                     0.88,
    "Skills – Speaking":                    0.45,
    "Skills – Mathematics":                 0.80,
    "Skills – Science":                     0.55,
    "Skills – Critical Thinking":           0.82,
    "Skills – Active Learning":             0.60,
    "Skills – Learning Strategies":         0.50,
    "Skills – Monitoring":                  0.45,
    "Skills – Social Perceptiveness":       0.30,
    "Skills – Coordination":                0.35,
    "Skills – Persuasion":                  0.40,
    "Skills – Negotiation":                 0.38,
    "Skills – Instructing":                 0.42,
    "Skills – Service Orientation":         0.35,
    "Skills – Complex Problem Solving":     0.80,
    "Skills – Operations Analysis":         0.70,
    "Skills – Technology Design":           0.55,
    "Skills – Programming":                 0.90,
    "Skills – Quality Control Analysis":    0.50,
    "Skills – Operations Monitoring":       0.35,
    "Skills – Troubleshooting":             0.45,
    "Skills – Systems Analysis":            0.72,
    "Skills – Systems Evaluation":          0.68,
    "Skills – Time Management":             0.45,
    "Skills – Management of Financial Resources": 0.55,
    "Skills – Management of Personnel Resources": 0.40,
    # ── Work Activities ────────────────────────────────────────────────────
    "Work Activities – Getting Information":              0.80,
    "Work Activities – Processing Information":           0.85,
    "Work Activities – Analyzing Data or Information":    0.82,
    "Work Activities – Making Decisions and Solving Problems": 0.75,
    "Work Activities – Thinking Creatively":              0.70,
    "Work Activities – Updating and Using Relevant Knowledge": 0.72,
    "Work Activities – Communicating with Supervisors, Peers, or Subordinates": 0.40,
    "Work Activities – Communicating with People Outside the Organization": 0.38,
    "Work Activities – Establishing and Maintaining Interpersonal Relationships": 0.25,
    "Work Activities – Developing Objectives and Strategies": 0.65,
    "Work Activities – Organizing, Planning, and Prioritizing Work": 0.55,
    "Work Activities – Performing Administrative Activities": 0.75,
    "Work Activities – Documenting/Recording Information": 0.80,
    "Work Activities – Interpreting the Meaning of Information for Others": 0.72,
    "Work Activities – Provide Consultation and Advice to Others": 0.60,
    # ── Knowledge ──────────────────────────────────────────────────────────
    "Knowledge – English Language":         0.85,
    "Knowledge – Mathematics":              0.80,   # kein Konflikt: Präfix unterscheidet
    "Knowledge – Computers and Electronics": 0.82,
    "Knowledge – Administration and Management": 0.60,
    "Knowledge – Economics and Accounting": 0.70,
    "Knowledge – Law and Government":       0.65,
    "Knowledge – Medicine and Dentistry":   0.45,
    "Knowledge – Education and Training":   0.55,
    "Knowledge – Psychology":               0.50,
    "Knowledge – Sociology and Anthropology": 0.55,
    "Knowledge – Philosophy and Theology":  0.60,
    "Knowledge – Engineering and Technology": 0.65,
    "Knowledge – Design":                   0.58,
}

# Rescaling auf [0.1, 0.9]
mu_min = min(mu_weights_raw.values())
mu_max = max(mu_weights_raw.values())

mu_weights = {
    k: 0.1 + (v - mu_min) / (mu_max - mu_min + 1e-9) * 0.8
    for k, v in mu_weights_raw.items()
}

def exposure_type(mu: float) -> str:
    if mu > 0.7:   return "substitution"
    if mu >= 0.3:  return "augmentation"
    return "negligible"

mu_df = pd.DataFrame([
    {"element_name": k, "mu_i": v, "exposure_type": exposure_type(v)}
    for k, v in mu_weights.items()
])

print(f"μ_i-Gewichte definiert: {len(mu_df)} Dimensionen")
print(mu_df["exposure_type"].value_counts().to_string())
print(f"\nμ_i-Bereich nach Rescaling: [{mu_df['mu_i'].min():.3f}, {mu_df['mu_i'].max():.3f}]")

mu_df.to_csv(PROCESSED / "mu_weights.csv", index=False)
print("\nGespeichert → data/processed/mu_weights.csv ✓")

---
## 5.2.3 Berechnung des Exposure-Index E_j

Der Index wird gemäss der in Kapitel 4.5 definierten Formel berechnet:

$$E_j = \sum_{i=1}^{n} \mu_i \cdot w_{ij} \cdot s_{ij}$$

**Terme:**
- $w_{ij}$ = Min-Max-normalisierter Importance-Wert aus O\*NET (Abschnitt 5.1.1)
- $\mu_i$ = LLM-Gewicht der Dimension i (Abschnitt 5.2.2)
- $s_{ij}$ = Skill-Overlap-Koeffizient: Pearson-Korrelation zwischen dem
  berufsspezifischen Fähigkeitsprofil ($w_{ij}$-Vektor) und dem LLM-Expositionsprofil ($\mu_i$-Vektor).
  Berufe mit $s_{ij} < 0.70$ erhalten Gewicht 0 (Cohen, 1988; Engeli, 2017).

**Subindizes:**
- $E^{sub}_j$: Dimensionen mit $\mu_i > 0.7$ (Substitutionspotenzial)
- $E^{aug}_j$: Dimensionen mit $0.3 \leq \mu_i \leq 0.7$ (Augmentationspotenzial)

In [ ]:
# Aktive Dimensionen (exposure != negligible) mit μ_i
mu_active = (
    mu_df[mu_df["exposure_type"] != "negligible"]
    .set_index("element_name")["mu_i"]
)

# Gemeinsame Dimensionen zwischen O*NET-Pivot und μ_i-Dictionary
common_dims = [c for c in onet_pivot.columns if c in mu_active.index]
print(f"Gemeinsame Fähigkeitsdimensionen: {len(common_dims)} von {len(mu_active)} definierten")

# O*NET-Matrix auf gemeinsame Dimensionen einschränken
onet_matrix = onet_pivot.set_index("soc_code")[common_dims].fillna(0)
mu_vector   = mu_active[common_dims].values

# LLM-Capability-Profil (normierter μ-Vektor als Referenz für s_ij)
llm_profile = mu_vector / (np.linalg.norm(mu_vector) + 1e-9)

# ── Skill-Overlap-Koeffizient s_ij ────────────────────────────────────────
def skill_overlap(row_vals: np.ndarray, reference: np.ndarray) -> float:
    """
    Pearson-Korrelation zwischen berufsspezifischem Fähigkeitsprofil
    und dem normierten LLM-Expositionsprofil.
    Gibt 0 zurück bei konstanten Profilen (keine Varianz).
    """
    if np.std(row_vals) < 1e-9:
        return 0.0
    r, _ = pearsonr(row_vals, reference)
    return max(r, 0.0)  # Negative Korrelationen → 0

sij_values = onet_matrix.apply(
    lambda row: skill_overlap(row.values, llm_profile), axis=1
)
# Binäre Maske: s_ij >= Schwellenwert → Beruf geht in Index ein
sij_mask = (sij_values >= SKILL_OVERLAP_THRESHOLD).astype(float)

print(f"Berufe mit s_ij >= {SKILL_OVERLAP_THRESHOLD}: "
      f"{int(sij_mask.sum())} von {len(sij_mask)}")

# ── E_j = Σ(μ_i * w_ij) * s_ij_mask ──────────────────────────────────────
weighted_sum = onet_matrix.dot(mu_vector)
E_j_raw      = weighted_sum * sij_mask

# Subindizes via np.where (vermeidet NaN-Propagation)
mu_vals  = mu_active[common_dims].values
mu_sub   = np.where(mu_vals > 0.7,                          mu_vals, 0.0)
mu_aug   = np.where((mu_vals >= 0.3) & (mu_vals <= 0.7),   mu_vals, 0.0)

E_sub_raw = onet_matrix.dot(mu_sub) * sij_mask
E_aug_raw = onet_matrix.dot(mu_aug) * sij_mask

# Min-Max-Normierung auf [0,1]
def minmax(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

index_df = pd.DataFrame({
    "soc_code":  onet_matrix.index,
    "s_ij":      sij_values.values,
    "E_j":       minmax(E_j_raw).values,
    "E_sub_j":   minmax(E_sub_raw).values,
    "E_aug_j":   minmax(E_aug_raw).values,
})

print(f"\nIndex berechnet: {len(index_df)} Berufe")
print(f"  E_j   – Mean: {index_df['E_j'].mean():.3f} | Std: {index_df['E_j'].std():.3f}")
print(f"  E^sub – Mean: {index_df['E_sub_j'].mean():.3f} | Std: {index_df['E_sub_j'].std():.3f}")
print(f"  E^aug – Mean: {index_df['E_aug_j'].mean():.3f} | Std: {index_df['E_aug_j'].std():.3f}")

### Monte-Carlo-Konfidenzintervall (Mapping-Konfidenz)

1000 Iterationen mit Gaussian-Noise auf den μ_i-Gewichten (σ=5%)  
zur Schätzung der Indexunsicherheit. Zielwert: ±8% (Raychaudhuri, 2008; vgl. Kap. 4.9).

In [ ]:
np.random.seed(RANDOM_SEED)
mc_results = []

for _ in range(N_MC_ITERATIONS):
    mu_noisy = mu_vector * (1 + np.random.normal(0, 0.05, size=len(mu_vector)))
    mu_noisy = np.clip(mu_noisy, 0.1, 0.9)
    E_mc = minmax(onet_matrix.dot(mu_noisy) * sij_mask)
    mc_results.append(E_mc.values)

mc_array  = np.array(mc_results)
ci_width  = np.percentile(mc_array, 97.5, axis=0) - np.percentile(mc_array, 2.5, axis=0)
mean_ci   = ci_width.mean() / 2 * 100  # in Prozent

print(f"Monte-Carlo ({N_MC_ITERATIONS} Iterationen):")
print(f"  Mittleres 95%-KI: ±{mean_ci:.1f}%")
print(f"  Zielwert ±8%: {'✓ erreicht' if mean_ci <= 8 else '⚠ überschritten → Sensitivitätsanalyse Kap. 6.5'}")

---
## 5.2.4 Vorbereitung der statistischen Analysen

Die aufbereiteten Datensätze werden für die drei Analysestufen (Kap. 4.8) strukturiert:
- **H1** (ANOVA): Gruppenunterschiede in E_j nach CH-ISCO-Hauptgruppe
- **H2** (Random Forest): Fähigkeitsdimensionen als Prädiktoren für E^sub_j
- **H3** (OLS): Beschäftigungsveränderung ΔBFS_j ~ E_j + Kontrollvariablen
- **Validierung**: Konvergente Validierung gegen ZHAW-Kuprecht-Index (2025)

In [ ]:
# ── Finales Sample: Index + Mapping + CH-ISCO + BFS zusammenführen ────────
ch_isco_merge = ch_isco[["ch_isco_4digit", "label", "main_group"]].copy()

final = (
    index_df
    .merge(train_mapping[["soc_code", "ch_isco_4digit", "mapping_stage"]],
           on="soc_code", how="inner")
    .merge(ch_isco_merge, on="ch_isco_4digit", how="left")
    .merge(bfs[["ch_isco_4digit", "employed_2022", "employed_2024",
                "delta_bfs", "is_outlier"]],
           on="ch_isco_4digit", how="left")
)

print(f"Finales Analysesample: {len(final)} Berufsgruppen")
print(f"  Hauptgruppe 1 (Führungskräfte):        {(final['main_group']==1).sum()}")
print(f"  Hauptgruppe 2 (Akademische Berufe):    {(final['main_group']==2).sum()}")
print(f"  Hauptgruppe 3 (Techniker etc.):        {(final['main_group']==3).sum()}")
print(f"  Davon mit ΔBFS_j-Daten:               {final['delta_bfs'].notna().sum()}")

In [ ]:
# ── H1-Datensatz: Gruppenunterschiede (ANOVA) ─────────────────────────────
h1 = final[["soc_code", "ch_isco_4digit", "main_group",
            "E_j", "E_sub_j", "E_aug_j"]].copy()
h1.to_csv(OUTPUT / "dataset_H1.csv", index=False)
print(f"H1: {len(h1)} Berufe → data/output/dataset_H1.csv ✓")

# ── H2-Datensatz: Feature Importance (Random Forest) ─────────────────────
# Alle O*NET-Dimensionen als Prädiktoren, E^sub_j als Kriteriumsvariable
h2 = final[["soc_code", "E_sub_j"]].merge(
    onet_pivot, on="soc_code", how="left"
)
h2.to_csv(OUTPUT / "dataset_H2.csv", index=False)
print(f"H2: {len(h2)} Berufe × {h2.shape[1]-2} Prädiktoren → data/output/dataset_H2.csv ✓")

# ── H3-Datensatz: OLS-Regression ΔBFS_j ~ E_j ────────────────────────────
h3 = final[["soc_code", "ch_isco_4digit", "main_group",
            "E_j", "delta_bfs", "is_outlier",
            "employed_2022", "employed_2024"]].copy()

# Qualifikationsniveau: Proxy via Hauptgruppe (ordinale Kodierung)
# HG 2 = höchste Qualifikation (Tertiär A), HG 1/3 = Tertiär B/Sek.II
h3["qualification_level"] = h3["main_group"].map({1: 3, 2: 4, 3: 3})

# Sektor: Platzhalter – manuelle Zuweisung nach Muster ch_isco_4digit
# Kategorien: ICT, Finanz/Beratung, Gesundheit/Bildung, Sonstiges
# → Diese Spalte muss manuell oder via Lookup-Tabelle befüllt werden
h3["sector"] = "TODO"  # Erinnerung: vor Analyse befüllen

h3.to_csv(OUTPUT / "dataset_H3.csv", index=False)
print(f"H3: {len(h3)} Berufe → data/output/dataset_H3.csv ✓")
print(f"  ⚠ Sektor-Variable noch als Platzhalter – vor Analyse befüllen")

# ── Standardisierter Fähigkeitsvektor (für Clusteranalyse in EDA) ─────────
skill_matrix = onet_pivot.set_index("soc_code")[common_dims]
skill_matrix_std = (skill_matrix - skill_matrix.mean()) / (skill_matrix.std() + 1e-9)
skill_matrix_std.to_csv(PROCESSED / "skill_vectors_standardized.csv")
print("Skill-Vektoren (standardisiert) → data/processed/skill_vectors_standardized.csv ✓")

# ── EDA-Gesamtdatensatz ───────────────────────────────────────────────────
final.to_csv(OUTPUT / "dataset_eda.csv", index=False)
print(f"EDA-Datensatz → data/output/dataset_eda.csv ✓")

In [ ]:
# ── Validierungs-Datensatz (Konvergente Validierung vs. Kuprecht 2025) ─────
validation = final[["soc_code", "ch_isco_4digit", "main_group",
                    "E_j", "E_sub_j", "E_aug_j"]].copy()
# Kuprecht-2025-Scores müssen manuell eingetragen werden
# Quelle: Kuprecht (2025), ZHAW Bachelor Thesis, Tabelle mit Substitutionswahrscheinlichkeiten
validation["kuprecht_2025_score"] = np.nan  # → nach Erhalt eintragen
validation.to_csv(OUTPUT / "dataset_validation.csv", index=False)
print(f"Validierungs-Datensatz → data/output/dataset_validation.csv ✓")
print(f"  ⚠ kuprecht_2025_score noch NaN – nach Erhalt der Daten eintragen")

---
## Zusammenfassung – Outputs dieses Notebooks

| Datei | Inhalt | Kapitel |
|-------|--------|---------|
| `processed/onet_pivot.csv` | Bereinigte O\*NET-Fähigkeitsmatrix (w_ij, normiert) | 5.1.1 |
| `processed/ch_isco_clean.csv` | CH-ISCO-19 Hauptgruppen 1–3 (harmonisiert) | 5.1.2 |
| `processed/bfs_clean.csv` | BFS ΔBFS_j 2022→2024, Ausreisser-Flag | 5.1.3 |
| `processed/onet_chisco_mapping.csv` | Vollständiges Mapping (Stufen 1–3) | 5.2.1 |
| `processed/stage3_manual_review.csv` | Export für manuelle Kodierung | 5.2.1 |
| `processed/train_mapping.csv` | Trainings-Mapping (80%) | 5.2.1 |
| `processed/holdout_mapping.csv` | Hold-out-Mapping (20%) | 5.2.1 |
| `processed/mu_weights.csv` | LLM-Gewichte μ_i (rescaled [0.1, 0.9]) | 5.2.2 |
| `processed/skill_vectors_standardized.csv` | Standardisierte Fähigkeitsvektoren | 5.2.4 |
| `output/dataset_eda.csv` | EDA-Gesamtdatensatz | 5.2.4 |
| `output/dataset_H1.csv` | H1: Gruppenunterschiede (ANOVA) | 5.2.4 |
| `output/dataset_H2.csv` | H2: Feature Importance (Random Forest) | 5.2.4 |
| `output/dataset_H3.csv` | H3: Beschäftigungsveränderung (OLS) | 5.2.4 |
| `output/dataset_validation.csv` | Konvergente Validierung vs. Kuprecht (2025) | 5.2.4 |

**Offene Punkte vor Kapitel 6:**
1. `stage3_manual_review.csv` durch zwei Kodierer befüllen → als `stage3_completed.csv` speichern → Notebook von Zelle 5.2.1 (Zusammenführen) neu ausführen
2. Sektor-Zuweisung in `dataset_H3.csv` ergänzen (Spalte `sector`)
3. Kuprecht-2025-Scores in `dataset_validation.csv` eintragen (Spalte `kuprecht_2025_score`)
4. Python- und R-Versionen in Kapitel 5.3.2 nachtragen sobald Umgebung fixiert